<a href="https://colab.research.google.com/github/Jewelzufo/SLM-Notebooks/blob/main/lfm25_230m_batch_review_sentiment_analysis_ipynb_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Batch review sentiment analysis with LFM2.5-230M

This beginner-friendly Colab notebook uses [LiquidAI/LFM2.5-230M](https://huggingface.co/LiquidAI/LFM2.5-230M), a compact instruction-tuned text-generation model, to classify five product reviews as **positive**, **negative**, or **neutral**.

The workflow is prompt-based: the model returns one label per review, then the notebook validates and summarizes those labels. The first run downloads the model files, so it may take a few minutes. A standard CPU Colab runtime works, although a GPU is faster.


## Goal

You will load an instruction-tuned Hugging Face model, analyze a batch of five reviews in one generation call, and validate the labels in a results table.

This is a practical demonstration, not a calibrated sentiment benchmark. Evaluate on representative labeled data before production use.


## 1. Install dependencies

Install a recent Transformers release because LFM2.5 is a recent model family.


In [ ]:
!pip -q install -U "transformers>=4.57.0" accelerate sentencepiece pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 26.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.


## 2. Import packages and select a device

The notebook uses a GPU when Colab provides one and otherwise runs on CPU.


In [ ]:
import json
import re
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "LiquidAI/LFM2.5-230M"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

Using device: cpu


## 3. Load LFM2.5-230M

Remote model code is enabled for compatibility with the model repository. Enable it only for repositories you trust.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype="auto", trust_remote_code=True
).to(DEVICE)
model.eval()

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Loaded {MODEL_ID}")

config.json:   0%|          | 0.00/1.57k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/489 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/4.73M [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/4.62k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/459M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/132 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/303 [00:00<?, ?B/s]

Loaded LiquidAI/LFM2.5-230M


## 4. Create five reviews

Replace these examples with your own reviews. The review ID lets the notebook reconnect every result to its source.


In [ ]:
reviews = [
    {"review_id": 1, "review": "The headphones sound fantastic, fit comfortably, and the battery lasted all week."},
    {"review_id": 2, "review": "The package arrived late and the screen has a dead pixel. I am disappointed."},
    {"review_id": 3, "review": "It does what the listing says. Setup took a few minutes, but nothing stood out."},
    {"review_id": 4, "review": "Excellent customer support replaced my broken unit quickly. I would buy again."},
    {"review_id": 5, "review": "The app crashes whenever I try to save a project, making the product unusable."},
]
reviews_df = pd.DataFrame(reviews)
reviews_df

,review_id,review
0,1,"The headphones sound fantastic, fit comfortabl..."
1,2,The package arrived late and the screen has a ...
2,3,It does what the listing says. Setup took a fe...
3,4,Excellent customer support replaced my broken ...
4,5,The app crashes whenever I try to save a proje...


## 5. Build one batch prompt

The model receives all five reviews together and is asked for a strict JSON array. Limiting the output to three labels makes automatic validation possible.


In [ ]:
allowed_labels = {"positive", "negative", "neutral"}
review_lines = "\n".join(
    f"{item['review_id']}. {item['review']}" for item in reviews
)
system_prompt = (
    "You are a careful sentiment-analysis assistant. "
    "Classify each review as positive, negative, or neutral. "
    "Return only valid JSON: an array of objects with exactly the keys "
    '"review_id" and "sentiment". Use lowercase sentiment labels.'
)
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": f"Reviews:\n{review_lines}"},
]
if getattr(tokenizer, "chat_template", None):
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
else:
    prompt = f"System: {system_prompt}\nUser: Reviews:\n{review_lines}\nAssistant:"
print(prompt)

<|startoftext|><|im_start|>system
You are a careful sentiment-analysis assistant. Classify each review as positive, negative, or neutral. Return only valid JSON: an array of objects with exactly the keys "review_id" and "sentiment". Use lowercase sentiment labels.<|im_end|>
<|im_start|>user
Reviews:
1. The headphones sound fantastic, fit comfortably, and the battery lasted all week.
2. The package arrived late and the screen has a dead pixel. I am disappointed.
3. It does what the listing says. Setup took a few minutes, but nothing stood out.
4. Excellent customer support replaced my broken unit quickly. I would buy again.
5. The app crashes whenever I try to save a project, making the product unusable.<|im_end|>
<|im_start|>assistant



## 6. Run batch inference

Sampling is disabled for repeatable results. The output token limit is intentionally small because the response is only five JSON objects.


In [ ]:
inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
with torch.inference_mode():
    generated_ids = model.generate(
        **inputs,
        max_new_tokens=160,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
new_tokens = generated_ids[0, inputs["input_ids"].shape[1]:]
raw_response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
print(raw_response)

```json
[
  {
    "review_id": 1,
    "sentiment": "positive"
  },
  {
    "review_id": 2,
    "sentiment": "negative"
  },
  {
    "review_id": 3,
    "sentiment": "neutral"
  },
  {
    "review_id": 4,
    "sentiment": "positive"
  },
  {
    "review_id": 5,
    "sentiment": "negative"
  }
]
```


## 7. Validate and display results

Generative models can occasionally add commentary or return malformed JSON. This helper extracts the JSON array and checks that all five IDs and labels match the requested schema.


In [ ]:
def parse_and_validate_sentiments(text, expected_ids):
    match = re.search(r"\[[\s\S]*\]", text)
    if not match:
        raise ValueError(f"No JSON array found in model response:\n{text}")
    try:
        rows = json.loads(match.group(0))
    except json.JSONDecodeError as exc:
        raise ValueError(f"Model returned invalid JSON:\n{text}") from exc
    if not isinstance(rows, list):
        raise ValueError("Expected a JSON array.")
    parsed = {}
    for row in rows:
        if set(row) != {"review_id", "sentiment"}:
            raise ValueError(f"Unexpected result schema: {row}")
        review_id = row["review_id"]
        sentiment = str(row["sentiment"]).lower().strip()
        if review_id not in expected_ids or sentiment not in allowed_labels:
            raise ValueError(f"Invalid result: {row}")
        if review_id in parsed:
            raise ValueError(f"Duplicate review_id: {review_id}")
        parsed[review_id] = sentiment
    if set(parsed) != set(expected_ids):
        raise ValueError(f"Expected IDs {expected_ids}; received {sorted(parsed)}")
    return parsed

sentiment_by_id = parse_and_validate_sentiments(
    raw_response, expected_ids=reviews_df["review_id"].tolist()
)
results = reviews_df.copy()
results["sentiment"] = results["review_id"].map(sentiment_by_id)
results

,review_id,review,sentiment
0,1,"The headphones sound fantastic, fit comfortabl...",positive
1,2,The package arrived late and the screen has a ...,negative
2,3,It does what the listing says. Setup took a fe...,neutral
3,4,Excellent customer support replaced my broken ...,positive
4,5,The app crashes whenever I try to save a proje...,negative


## 8. Summarize the five-review batch

This check confirms that every review received exactly one valid sentiment label.


In [ ]:
summary = (
    results["sentiment"].value_counts()
    .reindex(["positive", "neutral", "negative"], fill_value=0)
    .rename_axis("sentiment").reset_index(name="review_count")
)
display(summary)
assert len(results) == 5
assert results["sentiment"].isin(allowed_labels).all()
print("Validation passed: five reviews received valid sentiment labels.")

,sentiment,review_count
0,positive,2
1,neutral,1
2,negative,2


Validation passed: five reviews received valid sentiment labels.


## Next steps

- Replace the examples with a CSV column of your own reviews.
- Add a confidence or rationale only after testing whether the model returns it reliably.
- Measure quality against human-labeled examples before using the output for decisions.
